In [1]:
# =============================================================================
# COMPARISON EVALUATION: RL vs Baselines vs Metaheuristics
# =============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import time
import numpy as np
import pandas as pd
import torch

from data_manager import DataManager
from environment import Environment
from ddqn_agent import DDQNAgent

from all_methods import (
    simulate_route, VALID_PORTIONS,
    make_rl_selector, make_ns_selector, make_cs_selector,
    make_es_selector, make_rb_selector,
    run_alns, run_vns, run_ils, run_ga, run_sa, run_pso
)
from helper_functions import HelperFunctions

helper = HelperFunctions()

# =============================================================================
# CONFIGURATION
# =============================================================================

MAX_RL_STOPS = 3
MAX_FEASIBLE_STATIONS = 30
MODEL_PATH = "models/model_final"
DATASET_PATH = "test_dataset.json"
FLEET_PATH = "fleet.json"

# Methods to compare
METHODS = [
    'RL', 'NS', 'CS', 'ES', 'RB',
    'ALNS-50', 'ALNS-100',
    'VNS-50', 'VNS-100',
    'ILS-50', 'ILS-100',
    'GA-50', 'GA-100',
    'SA-50', 'SA-100',
    'PSO-30', 'PSO-50'
]


# =============================================================================
# ROUTE VALIDITY CHECK
# =============================================================================

def check_route_validity(env, dm, agent, inst_key, vehicle_id, max_stops, max_feasible_stations):
    """Check if route is valid for evaluation."""
    state = env.reset(inst_key, vehicle_id)
    
    if state is None:
        return False, 0
    
    stops = 0
    done = False
    
    while not done:
        feasible = state['station_ids']
        
        if len(feasible) > max_feasible_stations or len(feasible) == 0:
            return False, -1
        
        idx, p_idx = agent.select_action(state, training=False)
        station_id = feasible[min(idx, len(feasible) - 1)]
        portion = agent.charge_portions[min(p_idx, len(agent.charge_portions) - 1)]
        
        stops += 1
        if stops > max_stops:
            return False, -1
        
        state, _, done = env.step(station_id, portion)
    
    # Check depot return
    depot = dm.get_depot(inst_key)
    if depot:
        final_pos = env.current_position
        final_battery = env.current_battery
        dist_to_depot = helper.euclidean_distance(final_pos[0], final_pos[1], depot['x'], depot['y'])
        if final_battery < dist_to_depot:
            return False, -1
    
    return True, stops


# =============================================================================
# RUN SINGLE METHOD
# =============================================================================

def run_method(method, env, dm, agent, inst_key, vehicle_id):
    """Run a single method and return metrics + time."""
    start_time = time.time()
    
    if method == 'RL':
        selector = make_rl_selector(agent)
        result = simulate_route(env, dm, inst_key, vehicle_id, selector)
    elif method == 'NS':
        selector = make_ns_selector()
        result = simulate_route(env, dm, inst_key, vehicle_id, selector)
    elif method == 'CS':
        selector = make_cs_selector()
        result = simulate_route(env, dm, inst_key, vehicle_id, selector)
    elif method == 'ES':
        selector = make_es_selector()
        result = simulate_route(env, dm, inst_key, vehicle_id, selector)
    elif method == 'RB':
        selector = make_rb_selector()
        result = simulate_route(env, dm, inst_key, vehicle_id, selector)
    elif method == 'ALNS-50':
        result = run_alns(env, dm, inst_key, vehicle_id, iterations=50)
    elif method == 'ALNS-100':
        result = run_alns(env, dm, inst_key, vehicle_id, iterations=100)
    elif method == 'VNS-50':
        result = run_vns(env, dm, inst_key, vehicle_id, max_iterations=50)
    elif method == 'VNS-100':
        result = run_vns(env, dm, inst_key, vehicle_id, max_iterations=100)
    elif method == 'ILS-50':
        result = run_ils(env, dm, inst_key, vehicle_id, iterations=50)
    elif method == 'ILS-100':
        result = run_ils(env, dm, inst_key, vehicle_id, iterations=100)
    elif method == 'GA-50':
        result = run_ga(env, dm, inst_key, vehicle_id, population_size=50, generations=10)
    elif method == 'GA-100':
        result = run_ga(env, dm, inst_key, vehicle_id, population_size=100, generations=15)
    elif method == 'SA-50':
        result = run_sa(env, dm, inst_key, vehicle_id, iterations=50)
    elif method == 'SA-100':
        result = run_sa(env, dm, inst_key, vehicle_id, iterations=100)
    elif method == 'PSO-30':
        result = run_pso(env, dm, inst_key, vehicle_id, num_particles=30, iterations=50)
    elif method == 'PSO-50':
        result = run_pso(env, dm, inst_key, vehicle_id, num_particles=50, iterations=75)
    else:
        return None, 0
    
    elapsed = time.time() - start_time
    return result, elapsed


# =============================================================================
# MAIN
# =============================================================================

def run_comparison():
    print("=" * 70)
    print("COMPARISON: RL vs Baselines vs Metaheuristics")
    print("=" * 70)
    
    # Load
    print("\nLoading...")
    dm = DataManager(dataset_path=DATASET_PATH, fleet_path=FLEET_PATH)
    env = Environment(dm)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    agent = DDQNAgent(epsilon=0.0, device=device)
    agent.load(MODEL_PATH)
    agent.epsilon = 0.0
    
    # Get routes
    all_routes = []
    for inst_key, inst in dm.get_all_instances().items():
        for veh_key, veh in inst.get('vehicles', {}).items():
            veh_id = veh.get('id', veh_key)
            if veh.get('route') and len(veh['route']) > 2:
                all_routes.append((inst_key, veh_id))
    
    print(f"Total routes: {len(all_routes)}")
    
    # Filter
    print(f"\nFiltering (RL stops <= {MAX_RL_STOPS}, feasible stations <= {MAX_FEASIBLE_STATIONS})...")
    valid_routes = []
    
    for i, (inst_key, veh_id) in enumerate(all_routes):
        if (i + 1) % 100 == 0:
            print(f"  Checked {i + 1}/{len(all_routes)}...")
        try:
            is_valid, rl_stops = check_route_validity(env, dm, agent, inst_key, veh_id, MAX_RL_STOPS, MAX_FEASIBLE_STATIONS)
            if is_valid and rl_stops > 0:
                valid_routes.append((inst_key, veh_id))
        except:
            continue
    
    print(f"Valid routes: {len(valid_routes)}")
    
    if not valid_routes:
        print("No valid routes!")
        return None
    
    # ========== DEBUG: Test first route ==========
    print("\n" + "=" * 70)
    print("DEBUG: First route comparison")
    print("=" * 70)
    
    debug_inst, debug_veh = valid_routes[0]
    print(f"Route: {debug_inst} / {debug_veh}")
    
    vehicle, _ = dm.get_vehicle_data(debug_inst, debug_veh)
    route = vehicle.get('route', [])
    depot = dm.get_depot(debug_inst)
    stations = dm.get_stations(debug_inst)
    print(f"Route length: {len(route)} nodes")
    print(f"Depot: ({depot['x']}, {depot['y']})")
    print(f"Stations: {len(stations)}")
    print()
    
    for method in METHODS:
        result, elapsed = run_method(method, env, dm, agent, debug_inst, debug_veh)
        if result and result.get('feasible'):
            print(f"{method:8s}: total={result['total_cost']:8.1f}, charging={result['charging_cost']:8.1f}, "
                  f"stops={result['stops']}, battery@depot={result.get('battery_at_depot', '?'):6.1f}")
        else:
            bat = result.get('battery_at_depot', '?') if result else '?'
            print(f"{method:8s}: INFEASIBLE (battery@depot={bat})")
    
    print("=" * 70 + "\n")
    
    # Evaluate all routes
    print("Evaluating all routes...")
    
    feasibility_count = {m: 0 for m in METHODS}
    total_routes = len(valid_routes)
    all_route_results = []
    
    for i, (inst_key, veh_id) in enumerate(valid_routes):
        if (i + 1) % 2 == 0:
            print(f"  Route {i + 1}/{total_routes}...")
        
        route_results = {}
        for method in METHODS:
            try:
                result, elapsed = run_method(method, env, dm, agent, inst_key, veh_id)
                if result and result.get('feasible'):
                    route_results[method] = {
                        'travel': result['travel_cost'],
                        'charging': result['charging_cost'],
                        'waiting': result['waiting_cost'],
                        'total': result['total_cost'],
                        'stops': result['stops'],
                        'time': elapsed,
                        'battery_at_depot': result.get('battery_at_depot', 0)
                    }
                    feasibility_count[method] += 1
                else:
                    route_results[method] = None
            except:
                route_results[method] = None
        
        all_route_results.append(route_results)
    
    print("  Done!")
    
    # Shared feasible
    shared_feasible_indices = []
    for i, route_results in enumerate(all_route_results):
        if all(route_results.get(m) is not None for m in METHODS):
            shared_feasible_indices.append(i)
    
    print(f"\nShared feasible routes: {len(shared_feasible_indices)} / {total_routes}")
    
    # Calculate metrics
    results = {m: {'travel': [], 'charging': [], 'waiting': [], 'total': [], 'stops': [], 'time': [], 'battery_depot': []} for m in METHODS}
    
    for i in shared_feasible_indices:
        route_results = all_route_results[i]
        for method in METHODS:
            r = route_results[method]
            results[method]['travel'].append(r['travel'])
            results[method]['charging'].append(r['charging'])
            results[method]['waiting'].append(r['waiting'])
            results[method]['total'].append(r['total'])
            results[method]['stops'].append(r['stops'])
            results[method]['time'].append(r['time'])
            results[method]['battery_depot'].append(r['battery_at_depot'])
    
    # Summary
    print("\n" + "=" * 130)
    print(f"RESULTS SUMMARY (on {len(shared_feasible_indices)} shared feasible routes out of {total_routes} valid routes)")
    print("=" * 130)
    
    summary = []
    for method in METHODS:
        data = results[method]
        feasibility_pct = (feasibility_count[method] / total_routes) * 100
        
        if data['total']:
            summary.append({
                'Method': method,
                'Feasibility%': round(feasibility_pct, 1),
                'Travel': round(np.mean(data['travel']), 2),
                'Charging': round(np.mean(data['charging']), 2),
                'Waiting': round(np.mean(data['waiting']), 2),
                'Total': round(np.mean(data['total']), 2),
                'Stops': round(np.mean(data['stops']), 2),
                'Battery@Depot': round(np.mean(data['battery_depot']), 2),
                'Time(s)': round(np.mean(data['time']), 4)
            })
        else:
            summary.append({
                'Method': method,
                'Feasibility%': round(feasibility_pct, 1),
                'Travel': '-', 'Charging': '-', 'Waiting': '-',
                'Total': '-', 'Stops': '-', 'Battery@Depot': '-', 'Time(s)': '-'
            })
    
    df = pd.DataFrame(summary)
    print("\n" + df.to_string(index=False))
    
    df.to_csv('comparison_results.csv', index=False)
    print("\n\nSaved to comparison_results.csv")
    
    # Improvement analysis
    print("\n" + "=" * 70)
    print("IMPROVEMENT vs RL (on shared feasible routes)")
    print("=" * 70)
    
    rl_row = df[df['Method'] == 'RL']
    if not rl_row.empty and rl_row['Total'].values[0] != '-':
        rl_total = float(rl_row['Total'].values[0])
        print(f"\nRL Total: {rl_total}")
        for _, row in df.iterrows():
            if row['Method'] != 'RL' and row['Total'] != '-':
                diff = float(row['Total']) - rl_total
                pct = (diff / rl_total) * 100
                symbol = "+" if diff > 0 else ""
                winner = "RL wins" if diff > 0 else "baseline wins"
                print(f"  {row['Method']:8s}: {symbol}{diff:8.2f} ({symbol}{pct:5.2f}%) [{winner}]")
    
    return df


if __name__ == "__main__":
    df = run_comparison()

COMPARISON: RL vs Baselines vs Metaheuristics

Loading...
✓ Loaded dataset with 15 instances
✓ Loaded fleet data with 100 vehicles
✓ Computed normalization constants

DDQN Agent Initialization
Using device: cuda
GPU: NVIDIA RTX A5000
GPU Memory: 25.76 GB
CUDA Version: 12.1

Initializing DDQN Architecture on device: cuda
Using default vehicle input dimension: 7 (x, y, battery, max_battery, time, dist_next, remaining_dist)
✓ All networks successfully moved to cuda
Total trainable parameters: 114,023
All networks loaded from prefix: models/model_final
Total routes: 750

Filtering (RL stops <= 3, feasible stations <= 30)...
  Checked 100/750...
  Checked 200/750...
  Checked 300/750...
  Checked 400/750...
  Checked 500/750...
  Checked 600/750...
  Checked 700/750...
Valid routes: 124

DEBUG: First route comparison
Route: 16 / V3
Route length: 7 nodes
Depot: (250.0, 250.0)
Stations: 501

RL      : total=  8965.5, charging=  8562.1, stops=1, battery@depot=   4.6
NS      : total=  9668.3, c